# Custom Flood SatViT Training Workbook

## Step 0: Import necessary libraries and set up the environment

In [ ]:
import random

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from dataset import Sen1Floods11Dataset
from einops import rearrange
from models import FloodSegmentationHead, build_satvit_encoder
from torch.utils.data import DataLoader

In [ ]:
# Create folder paths for the dataset and CSV
data_folder = r'C:\Users\josep\Desktop\ICS\ICS 605\ICS605_sandbox_1\Final Project\data\sentinel_imagery\v1.1\data'

In [ ]:
# Define the exact paths to your CSV splits
train_csv = r'C:\Users\josep\Desktop\ICS\ICS 605\ICS605_sandbox_1\Final Project\data\sentinel_imagery\v1.1\splits\flood_handlabeled\flood_train_data.csv'
valid_csv = r'C:\Users\josep\Desktop\ICS\ICS 605\ICS605_sandbox_1\Final Project\data\sentinel_imagery\v1.1\splits\flood_handlabeled\flood_valid_data.csv'
test_csv = r'C:\Users\josep\Desktop\ICS\ICS 605\ICS605_sandbox_1\Final Project\data\sentinel_imagery\v1.1\splits\flood_handlabeled\flood_test_data.csv'

## Step 1: Create a Class that handles creating tensors out of the data set

## Step 2: SatViT

### Prepare the dataset, dataloader, and libraries for training

### SatViT Encoder

In [ ]:
# Determine if we have a GPU available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load the pre-trained SatViT encoder
satvit_model = build_satvit_encoder(device=device)

In [ ]:
# Load the Pre-trained Weights
# Using map_location to ensure it loads even if you are testing on a CPU
checkpoint_path = 'SatViT_V2.pt'
pretrained_checkpoint = torch.load(checkpoint_path, map_location=device)
satvit_model.load_state_dict(pretrained_checkpoint)

In [ ]:
# Partial Freezing (Fine-Tuning)

# First, freeze the entire model
for param in satvit_model.parameters():
    param.requires_grad = False
print('Pre-trained weights loaded and frozen!')

# Dynamically search for the last 2 layers (indexed 10 and 11) and unfreeze them
unfrozen_count = 0
for name, param in satvit_model.named_parameters():
    if 'encoder.layers.10' in name or 'encoder.layers.11' in name:
        param.requires_grad = True
        unfrozen_count += 1

print(f'Encoder partially unfrozen! ({unfrozen_count} parameter tensors activated for fine-tuning)')

## Step 3: Segmentation Decoder

In [ ]:
# Initialize the custom Decoder (Segmentation Head)
print('Initializing Segmentation Head...')
segmentation_decoder = FloodSegmentationHead(encoder_dim=768, num_classes=1).to(device)
print('Segmentation Head initialized successfully!')

## Step 4: THE TRAINING LOOP

### Define the loss function, hyperparameters, and optimizer for training the Decoder

In [ ]:
# Hyperparameters
num_epochs = 50
batch_size = 6  # Start small (4 or 8) so you don't run out of GPU memory
learning_rate = 1e-4

In [ ]:
# Create THREE separate datasets directly from the CSVs
print('Loading official dataset splits...')
train_dataset = Sen1Floods11Dataset(data_folder=data_folder, csv_path=train_csv)
val_dataset = Sen1Floods11Dataset(data_folder=data_folder, csv_path=valid_csv)
test_dataset = Sen1Floods11Dataset(data_folder=data_folder, csv_path=test_csv)
print(
    f'Total Images -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}'
)

In [ ]:
# Pass it to the PyTorch DataLoader
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# Note: We never shuffle Validation or Test sets, we want them in a consistent order!
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Define the Loss Function and Optimizer
# We use BCEWithLogitsLoss for Binary Segmentation (Flood vs. No Flood)
# It is mathematically more stable than adding a Sigmoid layer to the end of our decoder.
criterion = nn.BCEWithLogitsLoss()

In [ ]:
# We ONLY give the optimizer the parameters for our new Decoder and the last 2 encoder layers that we unfroze!
optimizer = optim.Adam(
    [
        {'params': satvit_model.encoder.layers[10].parameters(), 'lr': 1e-4},
        {'params': satvit_model.encoder.layers[11].parameters(), 'lr': 1e-4},
        {'params': segmentation_decoder.parameters(), 'lr': 1e-4},
    ]
)

In [ ]:
# For plotting
train_losses = []
val_losses = []

In [ ]:
# This will reduce the learning rate if the loss stops going down for 5 epochs in a row, which can help you get better results without manually tuning the learning rate.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

### Training Loop

In [ ]:
# The Training Loop
print('Starting training loop...')
# One epoch = one full pass through the entire training dataset (and then validation)
for epoch in range(num_epochs):
    # --- 1. TRAINING PHASE ---

    # Set the segmentation decoder & satvit_model to training mode for last 2 encoder layers, activates dropout and batchnorm layers if they exist
    satvit_model.train()
    segmentation_decoder.train()
    # Counter to accumulate training loss over the epoch
    running_train_loss = 0.0
    # Instead of feeding all images at once (which would crash the GPU),
    # it loads a small "batch" (e.g., 8 images and 8 answer keys) at a time.
    for batch_idx, (images, masks) in enumerate(train_loader):
        # Move data to the GPU (or CPU)
        images = images.to(device)
        # .float() ensures the masks are decimals, which the BCE Loss function requires.
        masks = masks.to(device).float()

        # --- Data Augmentation ---

        # Apply the same random flips to both images and masks to keep them aligned
        # 50% chance to flip horizontally
        if random.random() > 0.5:
            images = torch.flip(images, dims=[3])
            masks = torch.flip(masks, dims=[2])
        # 50% chance to flip vertically
        if random.random() > 0.5:
            images = torch.flip(images, dims=[2])
            masks = torch.flip(masks, dims=[1])

        # ---Forward Pass---

        # CRITICAL: Wipes the mathematical gradients from the PREVIOUS batch clean.
        # If you forget this, PyTorch will add the new math on top of the old math, ruining the model.
        optimizer.zero_grad()
        # Uses the 'einops' library to slice the 256x256 image into a sequence of tiny 8x8 squares.
        # Vision Transformers require data in sequence "patches", not as whole images.
        image_patches = rearrange(images, 'b c (h i) (w j) -> b (h w) (c i j)', i=8, j=8)
        # THE FORWARD PASS: Feeds the patches through the Transformer to extract complex features,
        # then passes those features into your custom Decoder to draw the flood map.
        encodings = satvit_model.encode(images_patches=image_patches)
        raw_predictions = segmentation_decoder(encodings)

        # ---Loss & Backprop---

        # Calculates how wrong the model's prediction was compared to the ground truth mask.
        # .unsqueeze(1) adds a blank "channel" dimension to the mask (Batch, 1, 256, 256)
        # so its shape perfectly matches the model's output.
        loss = criterion(raw_predictions, masks.unsqueeze(1))
        # THE BACKWARD PASS: Calculates the gradient (the exact direction and amount)
        # that every single weight in the network needs to change to get a better score next time.
        loss.backward()
        # The Optimizer (Adam) takes the gradients calculated above and actually updates/changes the weights.
        optimizer.step()
        # Plucks the raw number (.item()) out of the PyTorch loss tensor and adds it to our running total.
        running_train_loss += loss.item()

    # At the end of the epoch, calculates the average error across all batches and saves it for our graph.
    avg_train_loss = running_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- 2. VALIDATION PHASE (The "Quiz") ---

    # Tells PyTorch to put its "Test Taking Hat" on.
    # Turns off active learning features like Dropout so the model uses 100% of its "brainpower".
    satvit_model.eval()
    segmentation_decoder.eval()
    # reset the validation loss counter
    running_val_loss = 0.0

    # CRITICAL: Tells the GPU to completely stop tracking gradients.
    # We aren't learning here, just testing. This saves massive amounts of VRAM and computation time.
    with torch.no_grad():  # NO backpropagation allowed here!
        # The exact same sequence as the Forward Pass above, but testing on data the model has never seen.
        # Notice there is no optimizer.step() or loss.backward() here!
        for val_images, val_masks in val_loader:
            val_images = val_images.to(device)
            val_masks = val_masks.to(device).float()
            # Forward Pass only
            val_patches = rearrange(val_images, 'b c (h i) (w j) -> b (h w) (c i j)', i=8, j=8)
            val_encodings = satvit_model.encode(images_patches=val_patches)
            val_preds = segmentation_decoder(val_encodings)
            # Calculate validation loss
            v_loss = criterion(val_preds, val_masks.unsqueeze(1))
            running_val_loss += v_loss.item()

    # Averages the test loss, saves it for the graph, and prints the final report card for this epoch.
    avg_val_loss = running_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    print(
        f'Epoch [{epoch + 1}/{num_epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}'
    )

print('Training Complete! The model now knows how to find floods.')

In [ ]:
# Plot Training vs. Validation Loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss', color='blue')
plt.plot(val_losses, label='Validation Loss', color='orange')
plt.title('Model Learning Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss (BCE)')
plt.legend()
plt.show()

In [ ]:
# Save the model's "brain" to your hard drive!
torch.save(segmentation_decoder.state_dict(), 'final_project_decoder.pth')
torch.save(satvit_model.state_dict(), 'final_project_encoder.pth')

print('Training Complete. Both Encoder and Decoder Saved!')

## Step 5: VISUALIZATION & TESTING 

In [ ]:
# Put the decoder into "Evaluation" mode (turns off training-specific layers like Dropout)
segmentation_decoder.eval()

In [ ]:
# Grab a single random batch of data from our DataLoader
print('Fetching satellite imagery for inference...')
images, masks = next(iter(test_loader))

# Move to GPU/CPU
images = images.to(device)
masks = masks.to(device)

In [ ]:
# --- The Forward Pass ---
with torch.no_grad():  # No gradients needed, we are just testing!
    # Chop and Encode
    image_patches = rearrange(images, 'b c (h i) (w j) -> b (h w) (c i j)', i=8, j=8)
    encodings = satvit_model.encode(images_patches=image_patches)

    # Decode into a Flood Map
    raw_predictions = segmentation_decoder(encodings)

In [ ]:
# --- Post-Processing ---
# Because we used BCEWithLogitsLoss during training, the model outputs raw numbers (logits).
# We must use a Sigmoid function to squish them into percentages (0% to 100% chance of flood).
probabilities = torch.sigmoid(raw_predictions)

# Create a strict binary map: If the model is > 50% sure it's a flood, color it in (1).
binary_predictions = (probabilities > 0.5).float()

# Move the tensors back to the CPU so Matplotlib can read them
images_cpu = images.cpu().numpy()
masks_cpu = masks.cpu().numpy()
preds_cpu = binary_predictions.cpu().numpy()

In [ ]:
# --- Plotting ---
# We will plot the first image in the batch (index 0)
idx = 0

fig, arr = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1: The Raw Satellite Imagery
# (We use Channel 0, which is the Sentinel-1 VV Radar band, as our background)
arr[0].imshow(images_cpu[idx, 0, :, :], cmap='gray')
arr[0].set_title('Input: Sentinel-1 Radar', fontsize=14)
arr[0].axis('off')

# Plot 2: The Ground Truth (Answer Key)
arr[1].imshow(masks_cpu[idx], cmap='Blues', vmin=0, vmax=1)
arr[1].set_title('Ground Truth (Actual Flood)', fontsize=14)
arr[1].axis('off')

# Plot 3: The AI's Prediction
arr[2].imshow(preds_cpu[idx, 0], cmap='Blues', vmin=0, vmax=1)
arr[2].set_title('SatViT AI Prediction', fontsize=14)
arr[2].axis('off')

plt.tight_layout()
plt.show()

## Step 6: QUANTITATIVE EVALUATION (mIoU)

In [ ]:
# Creates function for calculating mIoU (Mean Intersection over Union) across the entire test set
def calculate_miou(loader, encoder, decoder, device):
    """
    Calculates the Intersection over Union (IoU) for the water class
    across an entire Dataloader.
    """
    # Put models in evaluation mode
    encoder.eval()
    decoder.eval()

    total_intersection = 0.0
    total_union = 0.0

    print('Calculating mIoU... Please wait.')

    with torch.no_grad():  # No training happening here!
        for batch_idx, (images, masks) in enumerate(loader):
            images = images.to(device)
            masks = masks.to(device).float()

            # --- Forward Pass ---
            image_patches = rearrange(images, 'b c (h i) (w j) -> b (h w) (c i j)', i=8, j=8)
            encodings = encoder.encode(images_patches=image_patches)
            raw_predictions = decoder(encodings)

            # --- Get Binary Map ---
            probabilities = torch.sigmoid(raw_predictions)
            # We squeeze out the channel dimension so preds shape matches masks shape
            preds = (probabilities > 0.5).squeeze(1).float()

            # --- The mIoU Math ---
            # Intersection: Pixels where BOTH prediction and mask are 1 (True Positives)
            intersection = (preds * masks).sum()

            # Union: Pixels where EITHER prediction OR mask is 1
            union = preds.sum() + masks.sum() - intersection

            total_intersection += intersection.item()
            total_union += union.item()

    # Calculate the final score, adding a tiny number (1e-6) to prevent dividing by zero
    final_miou = total_intersection / (total_union + 1e-6)

    return final_miou

In [ ]:
# Run the evaluation
checkpoint_miou = calculate_miou(test_loader, satvit_model, segmentation_decoder, device)

print('=====================================')
print(f'Final Model mIoU: {checkpoint_miou:.4f} ({checkpoint_miou * 100:.2f}%)')
print('=====================================')